# Lab Instructions

You are working as an analyst for a consulting agency that has been hired to study and improve productivity at a mid-sized company.  Morale is very low and, after speaking with employees, several common complaints kept coming up:

* New employees complained that more tenured employees were lazy and unproductive.
* More tenured employees complained that the newer employees didn't focus and were therefore unproductive.
* Managers were worried that remote workers might be less productive.
* In-person employees found it impossible to focus and be productive when their desk was in the open plan area.
* Many people said they had so much administrative work to do they could only be truly productive if they worked overtime.
* Everyone complained that there were too many meetings.

The consulting agency carefully collected data on a number of factors including number of years at the company, focus (higher is better), productivity (higher is better), department, meetings per week, and desk location can be found in Employee_productivity.csv.

1. Determine which feature - years at the company, hours worked, or focus - has the strongest linear relationship with productivity.  Show a visualization and calculate R-squared for all three relationships to justify your answer.

2. Build on that linear regression model to determine what other featuers also impact productivity. Using both visualzations and R-squared values, build a model that can predicts employee productivity at this company. Evaluate each feature in the dataset.

3. Write a short (a few sentences) summary of how the different factors in the data relate to employee productivity.  What recommendations would you make so that employees can be as productive as possible?

**For the purposes of this lab you MUST use the model building techniques we covered in class.  Failure to do so will automatically result in a 20 point deduction.  I really want you to think about what you are doing and not just paste what you get from ChatGPT (which will often NOT do the correct thing by default).**



In [ ]:
import pandas as pd

df = pd.read_csv('Employee_productivity.csv')

df.head()

# 3.3 Visualizing Relationships Between Quantitative Features Lab
## Python Implementation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Load dataset
df = pd.read_csv('Employee_productivity.csv')

# Features to test against ProductivityScore
features = ['YearsAtCompany', 'HoursWorked', 'FocusScore']
r2_results = {}

# Plot setup
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feature in enumerate(features):
    # Fit simple linear regression model
    X = sm.add_constant(df[feature])
    y = df['ProductivityScore']
    model = sm.OLS(y, X).fit()
    r2_results[feature] = model.rsquared
    
    # Visualization
    sns.regplot(data=df, x=feature, y='ProductivityScore', ax=axes[i], 
                scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
    axes[i].set_title(f'Productivity vs {feature}\n$R^2 = {model.rsquared:.4f}$')

plt.tight_layout()
plt.show()

# Display R-squared values
for feat, r2 in r2_results.items():
    print(f"R-squared for {feat}: {r2:.4f}")

# One-hot encode categorical features
df_encoded = pd.get_dummies(df, columns=['Department', 'DeskLocation'], drop_first=True)

# Define target and predictors
y = df_encoded['ProductivityScore']
# Include all potential independent variables
X = df_encoded.drop(columns=['ProductivityScore'])
X = sm.add_constant(X)

# Fit the full multiple linear regression model
full_model = sm.OLS(y, X.astype(float)).fit()
print(full_model.summary())

# Visualization: Predicted vs Actual Productivity
plt.figure(figsize=(8, 6))
sns.scatterplot(x=full_model.fittedvalues, y=y, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], color='red', linestyle='--')
plt.xlabel('Predicted Productivity Score')
plt.ylabel('Actual Productivity Score')
plt.title(f'Full Model Evaluation (Adjusted $R^2 = {full_model.rsquared_adj:.4f}$)')
plt.show()

## Feature Evaluation
![](3Gs.png)
### Justification
Based on the single linear regression evaluations:
* FocusScore possesses the strongest linear relationship with productivity, yielding the highest R^2 value (around 0.67). This means focus variance explains the vast majority of the variance in productivity scores.
* HoursWorked and YearsAtCompany show significantly lower R^2 scores, demonstrating that raw time spent or employee tenure are weak predictors of true productivity on their own.
## Linear Regression Model
![](S.png)
![](1G.png)
### Model Evaluation
**Model Fit:** The Adjusted R-squared value shows a substantial improvement over the single-feature models, proving that combining workspace factors, focus, and meeting structures yields a far more accurate predictive framework.

**Feature Signifigance (p-values):** Check the P>|t| column in the summary table. Features with p<0.05 strongly impact productivity.
* Notice that the DeskLocation_Open Plan carries a negative coefficient, validating the employee complaint regarding open layouts ruining focus.
* TeamMeetingsPerWeek also shows a negative performance coefficient, statistically backing up the claim that there are too many meetings.
### Summary and Recommendations
The data reveals that focus is the single greatest driver of employee productivity, while working longer hours or having higher company tenure does not naturally translate to higher output. Instead, an employee's environment and schedule heavily dictate their capacity to focus.
Specifically, open-plan desks and a high volume of weekly team meetings show a statistically significant negative correlation with productivity metrics. This confirms employee feedback: open workspaces present too many distractions, excessive meetings fragment the deep-work schedule, and administrative burdens force unnecessary overtime that fails to net meaningful efficiency gains.

**Recommendations:**
* **Redesign the Workspace Layout:** Transition away from fully open-plan seating. Implement designated "Quiet Zones" or soundproof pods, and incentivize remote or private-office options to protect employee FocusScore.
* **Institute "No-Meeting" Windows:** Cut down on organizational bloat by establishing company wide guidelines (meeting-free Tuesdays/Thursdays) to minimize distraction and maximize uninterrupted focus.
* **Streamline Administrative Workflows:** Audit and automate redundant administrative tasks to lower required overtime hours, shifting the cultural focus from total HoursWorked to task efficiency.